# Agent 1: bardzo prosty agent lokalny / niepłatny

Ten notebook pokazuje **najprostszy zewnętrzny agenta**, który:
- przyjmuje cel użytkownika,
- buduje prosty prompt,
- wywołuje **lokalny model przez Ollama** albo tryb `MockLLM`,
- zwraca odpowiedź.

Scenariusz dydaktyczny:
- minimalna architektura agenta,
- adapter do modelu,
- jedno wejście -> jedna decyzja -> jedna odpowiedź.

## Modele
Przykładowe lokalne modele do użycia przez Ollama:
- `gpt-oss:20b`
- `qwen2.5:7b`
- `llama3.2:3b`

Notebook uruchamia się także bez Ollama, w trybie **mock**, aby można było bezpiecznie pokazać logikę na zajęciach.

In [ ]:
!apt-get update -y
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,116 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3bu

In [ ]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [ ]:
import time
time.sleep(8)

In [ ]:
!ollama pull gemma2:2b

In [ ]:
USE_OLLAMA = True   # zmień na True, jeśli masz uruchomione Ollama lokalnie
OLLAMA_MODEL = "gemma2:2b"

In [ ]:
from dataclasses import dataclass
from typing import Protocol
import json
import textwrap

class LLMBackend(Protocol):
    def generate(self, prompt: str) -> str:
        ...

class MockLLM:
    def generate(self, prompt: str) -> str:
        prompt_lower = prompt.lower()
        if "mail" in prompt_lower:
            return (
                "Temat: Krótka aktualizacja\n\n"
                "Dzień dobry,\n"
                "przesyłam krótką informację, że prace postępują zgodnie z planem. "
                "Szczegóły mogę dosłać w osobnej wiadomości.\n\nPozdrawiam"
            )
        return (
            "To jest odpowiedź wygenerowana przez MockLLM. "
            "Agent rozpoznał zadanie i przygotował zwięzłą odpowiedź."
        )

class OllamaLLM:
    def __init__(self, model: str, url: str = "http://localhost:11434/api/generate"):
        self.model = model
        self.url = url

    def generate(self, prompt: str) -> str:
        import requests
        payload = {"model": self.model, "prompt": prompt, "stream": False}
        resp = requests.post(self.url, json=payload, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        return data["response"]

def get_llm(use_ollama: bool, model: str) -> LLMBackend:
    if use_ollama:
        return OllamaLLM(model=model)
    return MockLLM()

In [ ]:
@dataclass
class SimpleAgent:
    llm: LLMBackend
    system_role: str = "Jesteś pomocnym lokalnym agentem wykonującym pojedyncze zadanie."

    def run(self, user_goal: str) -> str:
        prompt = textwrap.dedent(f'''
        [SYSTEM]
        {self.system_role}

        [USER]
        Cel: {user_goal}

        Wykonaj zadanie krótko, jasno i praktycznie.
        ''').strip()
        return self.llm.generate(prompt)

In [ ]:
# Test 1
llm = get_llm(USE_OLLAMA, OLLAMA_MODEL)
agent = SimpleAgent(llm=llm)

result = agent.run("Napisz krótki profesjonalny mail z informacją o postępie prac.")
print(result)
assert isinstance(result, str) and len(result) > 20

## Jak uruchomić z prawdziwym lokalnym modelem

1. Zainstaluj Ollama.
2. Pobierz model, np.:
```bash
ollama pull qwen2.5:7b
```
3. Ustaw:
```python
USE_OLLAMA = True
OLLAMA_MODEL = "qwen2.5:7b"
```
4. Uruchom notebook ponownie.